# Person 2 — Text Preprocessing

Tokenization, lowercasing, stopword removal, stemming and lemmatization, with before/after examples.
The input is CV or job text that Person 1's `clean_text` has already cleaned.

In [1]:
# Find the project folder whether this notebook runs from the
# project root or from the notebooks folder.
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "samples").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

In [2]:
# Download the NLTK word lists once (stopwords and WordNet)
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

True

In [3]:
import pandas as pd
from cv_job_matching import clean_text, load_jobs, read_cv
from cv_job_matching.preprocessing import (
    lemmatize_tokens, preprocess, preprocess_jobs, preprocessing_steps,
    remove_stopwords, stem_tokens, to_lowercase, tokenize,
)

## 1. Tokenization

Split text into words. The regular expression keeps technical names whole: `C++`, `C#`, `.NET`, `ASP.NET`, `Node.js`. Punctuation is dropped.

In [4]:
sample = "Built web apps using C#, ASP.NET Core and .NET; managed SQL databases in 2025."
tokens = tokenize(sample)
print(tokens)

['Built', 'web', 'apps', 'using', 'C#', 'ASP.NET', 'Core', 'and', '.NET', 'managed', 'SQL', 'databases', 'in', '2025']


In [5]:
# Compare: a plain split on spaces keeps punctuation glued to the words
print(sample.split())

['Built', 'web', 'apps', 'using', 'C#,', 'ASP.NET', 'Core', 'and', '.NET;', 'managed', 'SQL', 'databases', 'in', '2025.']


## 2. Lowercasing

`Python`, `PYTHON` and `python` should count as the same word.

In [6]:
lowercase = to_lowercase(tokens)
print(lowercase)

['built', 'web', 'apps', 'using', 'c#', 'asp.net', 'core', 'and', '.net', 'managed', 'sql', 'databases', 'in', '2025']


## 3. Stopword removal

NLTK's English list (198 words such as *the, and, with, for*). Number-only tokens like `2025` are dropped too.

In [7]:
filtered = remove_stopwords(lowercase)
print(filtered)
print(f"{len(lowercase)} tokens -> {len(filtered)} tokens")

['built', 'web', 'apps', 'using', 'c#', 'asp.net', 'core', '.net', 'managed', 'sql', 'databases']
14 tokens -> 11 tokens


## 4. Stemming vs. lemmatization

- **Stemming** (Porter) cuts word endings by rules. Fast, but roots are not always real words.
- **Lemmatization** (WordNet) looks words up in a dictionary and returns a real word.

Each word is tried as a noun first, then as a verb.

In [8]:
words = ["developing", "managed", "databases", "applications", "analyzing",
         "responsibilities", "engineering", "skills", "built", "pandas", "c++"]
pd.DataFrame({
    "word": words,
    "stemmed": stem_tokens(words),
    "lemmatized": lemmatize_tokens(words),
})

,word,stemmed,lemmatized
0,developing,develop,develop
1,managed,manag,manage
2,databases,databas,database
3,applications,applic,application
4,analyzing,analyz,analyze
5,responsibilities,respons,responsibility
6,engineering,engin,engineer
7,skills,skill,skill
8,built,built,build
9,pandas,pandas,pandas


## 5. Before / after table

Every stage for a few fictional CV sentences.

In [9]:
examples = [
    "Developed REST APIs with Python and Flask; managed PostgreSQL databases.",
    "The candidate is responsible for analyzing data and creating dashboards.",
    "Built web applications using C#, ASP.NET Core and .NET for 3 clients in 2025.",
]
rows = []
for sentence in examples:
    steps = preprocessing_steps(sentence)
    rows.append({
        "before": steps["original"],
        "after stopwords": " ".join(steps["no_stopwords"]),
        "stemmed": " ".join(steps["stemmed"]),
        "lemmatized": " ".join(steps["lemmatized"]),
    })
pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

,before,after stopwords,stemmed,lemmatized
0,Developed REST APIs with Python and Flask; managed PostgreSQL databases.,developed rest apis python flask managed postgresql databases,develop rest api python flask manag postgresql databas,develop rest api python flask manage postgresql database
1,The candidate is responsible for analyzing data and creating dashboards.,candidate responsible analyzing data creating dashboards,candid respons analyz data creat dashboard,candidate responsible analyze data create dashboard
2,"Built web applications using C#, ASP.NET Core and .NET for 3 clients in 2025.",built web applications using c# asp.net core .net clients,built web applic use c# asp.net core .net client,build web application use c# asp.net core .net client


## 6. The full CV

`preprocess()` runs all steps and returns one string, ready for TF-IDF (Person 4).

In [10]:
cv_text = clean_text(read_cv(ROOT / "samples" / "sample_cv.txt"))
processed_cv = preprocess(cv_text)

print("BEFORE:", cv_text[:400], "...\n")
print("AFTER: ", processed_cv[:400], "...\n")
print(f"Words before: {len(cv_text.split())}   Tokens after: {len(processed_cv.split())}")

BEFORE: SAMPLE CV - JUNIOR SOFTWARE DEVELOPER ===================================== ALEX MORGAN Beirut, Lebanon Phone: +961 XX XXX XXX Email: LinkedIn: linkedin.com/in/your-profile GitHub: github.com/your-username PROFESSIONAL SUMMARY -------------------- Motivated junior software developer with experience building web applications, working with relational databases, and solving problems using clean, main ...

AFTER:  sample cv junior software developer alex morgan beirut lebanon phone xx xxx xxx email linkedin linkedin.com profile github github.com username professional summary motivate junior software developer experience build web application work relational database solve problem use clean maintainable code interest full stack development cloud technology artificial intelligence education bachelor science c ...

Words before: 281   Tokens after: 221


## 7. All job descriptions

`preprocess_jobs()` adds a `processed_text` column. `rank_jobs` in `matching.py` reads that column first.

In [11]:
jobs = preprocess_jobs(load_jobs())
jobs["words_before"] = jobs["job_text"].str.split().str.len()
jobs["tokens_after"] = jobs["processed_text"].str.split().str.len()
print(f"{len(jobs)} jobs")
print(f"Average words before: {jobs['words_before'].mean():.0f}")
print(f"Average tokens after: {jobs['tokens_after'].mean():.0f}")
jobs[["Job title", "words_before", "tokens_after"]].head()

389 jobs
Average words before: 578
Average tokens after: 387


,Job title,words_before,tokens_after
0,Mechanical Engineer-Precision Engineering & Diamond Turning,574,415
1,Product design engineer- watch,316,208
2,Sr Controls/Electrical Engineer,497,320
3,"Launch Engineer, Fluid Systems",412,270
4,Manufacturing Process Engineer,320,238


In [12]:
print(jobs.loc[0, "processed_text"][:500])

job title mechanical engineer precision engineer diamond turn description open precision mechanical engineer support project national ignition facility photon science directorate nif p provide solution wide variety research development precision engineer project involve deliverable hardware design develop advance structural thermal fluid analytical tool part multidisciplinary team responsible ensure sustainment nif mission critical capability finish crystal optic use precision diamond turn syste


## Limits

- `IT` (information technology) becomes `it`, which is a stopword, so it is removed.
- Lemmatization without a part-of-speech tagger can choose the wrong form (for example `meeting` -> `meet`).
- Hyphenated names are split: `scikit-learn` -> `scikit`, `learn`.
- Stemming makes roots like `manag` and `databas`, which are fine for matching but hard to read; that is why lemmatization is the default.